In [2]:
import os 
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_columns', 500)
import numpy as np
import geopandas as gpd

In [3]:
template = pd.read_csv("D:\Work\WB\LDT\countries\SRB\datasets\RS_processed-data_full-indicator-list.csv".replace('\\', '/'))
template.columns

Index(['GID_2', 'NAME_2', 'year', 'hospital_accesibility',
       'school_accesibility', 'pm25_concentration', 'o3_concentration',
       'co2e_20yr_emissions_quantity', 'co2e_20yr_emissions_factor',
       'agriculture_emissions', 'forestry-and-land-use_emissions',
       'total_powerplant_coal_emissions_quantity',
       'assets_broadband_speed_target', 'avg_d_mbps_broadband',
       'key_structures_without_internet', 'healthcare-facilities-diversity',
       'luminosity', 'avg_d_mbps_mobile', 'railway_length_flood_risk',
       'road_length_flood_risk', 'population_sum', 'rail_length_flood_pcap',
       'road_length_flood_pcap', 'rail_length_heatwave_risk',
       'road_length_heatwave_risk', 'rail_length_pcap_heatwave',
       'road_length_pcap_heatwave'],
      dtype='object')

In [163]:
hospital = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_access_tohealthcarefacilities_full.csv").drop(['NAME_2'], axis=1)
hospital = hospital.fillna(0)
hospital = hospital.rename({'population_with_access': 'population_with_access_hospital'}, axis=1)
healthcare_diversity = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_healthcarefacilities_diversity_full.csv").drop(['NAME_2'], axis=1)
healthcare_diversity = healthcare_diversity.fillna(0)
school = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_access_toschool_full.csv").drop(['NAME_2'], axis=1)
school = school.fillna(0)
school = school.rename({'population_with_access': 'population_with_access_school'}, axis=1)
air = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_air_pollution_full.csv")
methane_full = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_ch4_emissions_full.csv").drop(['NAME_2'], axis=1)
methane_full = methane_full.rename({'emissions_quantity': 'ch4_emissions_quantity',
                              'emissions_factor': 'ch4_emissions_factor'}, axis=1)
co2e_full = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_co2e_emissions_full.csv").drop(['NAME_2'], axis=1)
co2e_full = co2e_full.rename({'emissions_quantity': 'co2e_20yr_emissions_quantity',
                              'emissions_factor': 'co2e_20yr_emissions_factor'}, axis=1)
co2e_sector = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_co2e_emissions_sector_full.csv").drop(['NAME_2'], axis=1)
coal_full = pd.read_csv(r"D:/Work/WB/LDT/countries/SRB/datasets/SRB_co2e_emissions_coal_full.csv").drop(['NAME_2'], axis=1)
fixed = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_fixed_full.csv").drop(['NAME_2'], axis=1)
fixed = fixed.rename({'avg_d_mbps': 'avg_d_mbps_broadband',
                      'null_perc': 'key_structures_without_internet'}, axis=1)
mobile = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_mobile_full.csv").drop(['NAME_2'], axis=1)
mobile = mobile.rename({'avg_d_mbps': 'avg_d_mbps_mobile'}, axis=1)
luminosity = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_luminosity_full.csv").drop(['geometry', 'ym'], axis=1)
flood = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_flood_risk_full.csv")[['GID_2', 'year', 'rail_length_flood_risk', 'road_length_flood_risk', 'rail_length_flood_pcap', 'road_length_flood_pcap']]
heat = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_heatwave_risk_full.csv").drop(['population_sum'], axis=1)
heat = heat.rename({'rail_length_km': 'rail_length_heatwave_risk', 
                    'road_length_km': 'road_length_heatwave_risk',}, axis=1)

years = [2021, 2022, 2023, 2024]
length_hospital = len(hospital)
hospital = hospital.loc[hospital.index.repeat(4)].reset_index(drop=True)
hospital['year'] = np.tile(years, length_hospital)

healthcare_diversity = healthcare_diversity.loc[healthcare_diversity.index.repeat(4)].reset_index(drop=True)
healthcare_diversity['year'] = np.tile(years, length_hospital)

school = school.loc[school.index.repeat(4)].reset_index(drop=True)
school['year'] = np.tile(years, length_hospital)

In [164]:
merge_cols = ['GID_2', 'year']
total = pd.merge(hospital, healthcare_diversity, on=merge_cols, how='left')
total = pd.merge(total, school.drop(['population_total'], axis=1), on=merge_cols, how='left')
total = pd.merge(total, air, on=merge_cols, how='left')
total = pd.merge(total, methane_full, on=merge_cols, how='left')
total = pd.merge(total, co2e_full, on=merge_cols, how='left')
total = pd.merge(total, co2e_sector, on=merge_cols, how='left')
total = pd.merge(total, coal_full, on=merge_cols, how='left')
total = pd.merge(total, fixed, on=merge_cols, how='left')
total = pd.merge(total, mobile, on=merge_cols, how='left')
total = pd.merge(total, luminosity, on=merge_cols, how='left')
total = pd.merge(total, flood, on=merge_cols, how='left')
total = pd.merge(total, heat, on=merge_cols, how='left')
total = total.drop_duplicates(subset=['GID_2', 'year'], keep='first').reset_index(drop=True)
 
total = total.fillna(0)
total = total.replace([np.inf, -np.inf], 0)

In [165]:
total['year'].value_counts()

year
2021    161
2022    161
2023    161
2024    161
Name: count, dtype: int64

In [181]:
poly = gpd.read_file(r"D:\Work\WB\LDT\countries\SRB\shapefiles\gadm41_SRB_2.json")

total = pd.merge(poly[['NAME_2', 'GID_2']], total, on=['GID_2'], how='right')
total


,NAME_2,GID_2,population_with_access_hospital,population_total,hospital_accesibility,year,healthcare-facilities-diversity,population_with_access_school,school_accesibility,pm25_concentration,pm10_concentration,no2_concentration,ch4_emissions_quantity,ch4_emissions_factor,co2e_20yr_emissions_quantity,co2e_20yr_emissions_factor,agriculture_emissions,buildings_emissions,forestry-and-land-use_emissions,fossil-fuel-operations_emissions,manufacturing_emissions,mineral-extraction_emissions,power_emissions,transportation_emissions,waste_emissions,total_powerplant_coal_emissions_quantity,avg_d_mbps_broadband,key_structures_without_internet,avg_d_mbps_mobile,luminosity,rail_length_flood_risk,road_length_flood_risk,rail_length_flood_pcap,road_length_flood_pcap,rail_length_heatwave_risk,road_length_heatwave_risk,rail_length_pcap_heatwave,road_length_pcap_heatwave
0,Bor,SRB.1.1_1,27642.516381,40536.546387,68.0,2021,1.388204,27243.418347,67.0,8.339596,11.149546,2.653598,267.411777,0.000310,1.014963e+06,0.844349,32676.431524,24958.90128,420892.892990,0.0,0.0,421417.14488,0.0,115017.588823,0.00000,0.0,56.760213,11.111111,32.129829,8406.669981,0.0,0.0000,0.0,0.000,90.178562,2091.458435,2.441314,56.619964
1,Bor,SRB.1.1_1,27642.516381,40536.546387,68.0,2022,1.388204,27243.418347,67.0,8.545606,10.125109,2.486569,270.543667,0.000310,2.528427e+05,0.874867,28250.330744,24929.13081,-681774.237670,0.0,0.0,751682.52020,0.0,129754.907749,0.00000,0.0,52.114375,0.000000,36.962571,4614.979995,0.0,0.0000,0.0,0.000,90.177980,2267.766246,2.441298,61.392969
2,Bor,SRB.1.1_1,27642.516381,40536.546387,68.0,2023,1.388204,27243.418347,67.0,8.003265,9.656753,2.532644,270.248959,0.000310,5.614135e+05,0.869124,28303.065777,25212.05418,-523748.188768,0.0,0.0,898776.07612,0.0,132870.472610,0.00000,0.0,70.101850,0.000000,43.568551,6226.329994,0.0,0.0000,0.0,0.000,90.177980,2507.180571,2.441298,67.874394
3,Bor,SRB.1.1_1,27642.516381,40536.546387,68.0,2024,1.388204,27243.418347,67.0,8.199900,8.778352,0.186570,271.194468,0.000310,5.600742e+05,0.869124,28303.065777,25212.05418,-523748.188768,0.0,0.0,898776.07612,0.0,131531.202531,0.00000,0.0,99.311692,0.000000,59.829482,9653.610001,0.0,0.0000,0.0,0.000,90.177980,2560.840628,2.441298,69.327079
4,Kladovo,SRB.1.2_1,0.000000,17511.680623,0.0,2021,0.000000,0.000000,0.0,8.230006,10.621915,3.599601,121.539696,0.000420,7.234743e+05,0.901010,17839.898915,20702.65185,624494.550810,0.0,0.0,0.00000,0.0,60437.161936,0.00000,0.0,46.579333,0.000000,26.880468,3257.810001,0.0,206235.5369,0.0,12.404,0.000000,1193.493364,0.000000,71.784655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,Ivanjica,SRB.9.3_1,0.000000,26057.588181,0.0,2024,0.000000,2267.348548,9.0,6.867846,7.880904,-0.404043,477.899229,0.003197,-1.466550e+05,0.867188,11867.729169,17856.98649,-490294.746010,0.0,0.0,0.00000,0.0,287911.326311,26003.72919,0.0,0.000000,0.000000,44.626340,4765.720001,0.0,0.0000,0.0,0.000,0.000000,3260.747844,0.000000,144.829915
640,Lučani,SRB.9.4_1,114.263064,18459.399899,1.0,2021,0.000000,645.637743,3.0,8.496553,12.604442,2.623166,107.097148,0.000420,3.203621e+05,0.987911,13450.592869,13186.14231,179835.382273,0.0,0.0,0.00000,0.0,113889.966026,0.00000,0.0,0.000000,0.000000,37.369589,4607.030001,0.0,0.0000,0.0,0.000,14.831077,1400.087125,0.938287,88.576439
641,Lučani,SRB.9.4_1,114.263064,18459.399899,1.0,2022,0.000000,645.637743,3.0,8.302285,10.655915,2.707540,108.632973,0.000420,-2.776789e+04,1.025088,11748.211498,13256.57226,-188004.022600,0.0,0.0,0.00000,0.0,135231.343915,0.00000,0.0,0.000000,0.000000,32.758064,4264.489998,0.0,0.0000,0.0,0.000,14.831077,1619.994138,0.938287,102.488844
642,Lučani,SRB.9.4_1,114.263064,18459.399899,1.0,2023,0.000000,645.637743,3.0,7.385581,9.693388,2.556668,108.824211,0.000420,-2.607153e+05,1.027252,11862.356688,13344.10959,-432489.695400,0.0,0.0,0.00000,0.0,146567.977617,0.00000,0.0,0.000000,0.000000,41.89040

In [182]:
total.to_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_absolute_full.csv", index=False)

In [180]:
total

,GID_2,population_with_access_hospital,population_total,hospital_accesibility,year,healthcare-facilities-diversity,population_with_access_school,school_accesibility,pm25_concentration,pm10_concentration,no2_concentration,ch4_emissions_quantity,ch4_emissions_factor,co2e_20yr_emissions_quantity,co2e_20yr_emissions_factor,agriculture_emissions,buildings_emissions,forestry-and-land-use_emissions,fossil-fuel-operations_emissions,manufacturing_emissions,mineral-extraction_emissions,power_emissions,transportation_emissions,waste_emissions,total_powerplant_coal_emissions_quantity,avg_d_mbps_broadband,key_structures_without_internet,avg_d_mbps_mobile,luminosity,rail_length_flood_risk,road_length_flood_risk,rail_length_flood_pcap,road_length_flood_pcap,rail_length_heatwave_risk,road_length_heatwave_risk,rail_length_pcap_heatwave,road_length_pcap_heatwave
0,SRB.1.1_1,27642.516381,40536.546387,68.0,2021,1.388204,27243.418347,67.0,8.339596,11.149546,2.653598,267.411777,0.000310,1.014963e+06,0.844349,32676.431524,24958.90128,420892.892990,0.0,0.0,421417.14488,0.0,115017.588823,0.00000,0.0,56.760213,11.111111,32.129829,8406.669981,0.0,0.0000,0.0,0.000,90.178562,2091.458435,2.441314,56.619964
1,SRB.1.1_1,27642.516381,40536.546387,68.0,2022,1.388204,27243.418347,67.0,8.545606,10.125109,2.486569,270.543667,0.000310,2.528427e+05,0.874867,28250.330744,24929.13081,-681774.237670,0.0,0.0,751682.52020,0.0,129754.907749,0.00000,0.0,52.114375,0.000000,36.962571,4614.979995,0.0,0.0000,0.0,0.000,90.177980,2267.766246,2.441298,61.392969
2,SRB.1.1_1,27642.516381,40536.546387,68.0,2023,1.388204,27243.418347,67.0,8.003265,9.656753,2.532644,270.248959,0.000310,5.614135e+05,0.869124,28303.065777,25212.05418,-523748.188768,0.0,0.0,898776.07612,0.0,132870.472610,0.00000,0.0,70.101850,0.000000,43.568551,6226.329994,0.0,0.0000,0.0,0.000,90.177980,2507.180571,2.441298,67.874394
3,SRB.1.1_1,27642.516381,40536.546387,68.0,2024,1.388204,27243.418347,67.0,8.199900,8.778352,0.186570,271.194468,0.000310,5.600742e+05,0.869124,28303.065777,25212.05418,-523748.188768,0.0,0.0,898776.07612,0.0,131531.202531,0.00000,0.0,99.311692,0.000000,59.829482,9653.610001,0.0,0.0000,0.0,0.000,90.177980,2560.840628,2.441298,69.327079
4,SRB.1.2_1,0.000000,17511.680623,0.0,2021,0.000000,0.000000,0.0,8.230006,10.621915,3.599601,121.539696,0.000420,7.234743e+05,0.901010,17839.898915,20702.65185,624494.550810,0.0,0.0,0.00000,0.0,60437.161936,0.00000,0.0,46.579333,0.000000,26.880468,3257.810001,0.0,206235.5369,0.0,12.404,0.000000,1193.493364,0.000000,71.784655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,SRB.9.3_1,0.000000,26057.588181,0.0,2024,0.000000,2267.348548,9.0,6.867846,7.880904,-0.404043,477.899229,0.003197,-1.466550e+05,0.867188,11867.729169,17856.98649,-490294.746010,0.0,0.0,0.00000,0.0,287911.326311,26003.72919,0.0,0.000000,0.000000,44.626340,4765.720001,0.0,0.0000,0.0,0.000,0.000000,3260.747844,0.000000,144.829915
640,SRB.9.4_1,114.263064,18459.399899,1.0,2021,0.000000,645.637743,3.0,8.496553,12.604442,2.623166,107.097148,0.000420,3.203621e+05,0.987911,13450.592869,13186.14231,179835.382273,0.0,0.0,0.00000,0.0,113889.966026,0.00000,0.0,0.000000,0.000000,37.369589,4607.030001,0.0,0.0000,0.0,0.000,14.831077,1400.087125,0.938287,88.576439
641,SRB.9.4_1,114.263064,18459.399899,1.0,2022,0.000000,645.637743,3.0,8.302285,10.655915,2.707540,108.632973,0.000420,-2.776789e+04,1.025088,11748.211498,13256.57226,-188004.022600,0.0,0.0,0.00000,0.0,135231.343915,0.00000,0.0,0.000000,0.000000,32.758064,4264.489998,0.0,0.0000,0.0,0.000,14.831077,1619.994138,0.938287,102.488844
642,SRB.9.4_1,114.263064,18459.399899,1.0,2023,0.000000,645.637743,3.0,7.385581,9.693388,2.556668,108.824211,0.000420,-2.607153e+05,1.027252,11862.356688,13344.10959,-432489.695400,0.0,0.0,0.00000,0.0,146567.977617,0.00000,0.0,0.000000,0.000000,41.890408,1914.559998,0.0,0.0000,0.0,0.000,14.800398,1866.055030,0.936346

In [171]:
keep_cols = ['GID_2', 'year', 'hospital_accesibility',
       'school_accesibility', 'pm25_concentration', 'pm10_concentration', 'no2_concentration',
       'co2e_20yr_emissions_quantity', 'co2e_20yr_emissions_factor', 'ch4_emissions_quantity',
       'agriculture_emissions', 'forestry-and-land-use_emissions',
       'total_powerplant_coal_emissions_quantity', 'avg_d_mbps_broadband',
       'key_structures_without_internet', 'healthcare-facilities-diversity',
       'luminosity', 'avg_d_mbps_mobile', 'rail_length_flood_risk',
       'road_length_flood_risk', 'rail_length_flood_pcap',
       'road_length_flood_pcap', 'rail_length_heatwave_risk',
       'road_length_heatwave_risk', 'rail_length_pcap_heatwave',
       'road_length_pcap_heatwave']

df = total[keep_cols]
df = df.drop(['co2e_20yr_emissions_factor',
       'agriculture_emissions', 'forestry-and-land-use_emissions',
       'rail_length_flood_pcap',
       'road_length_flood_pcap',
       'rail_length_pcap_heatwave',
       'road_length_pcap_heatwave'], axis=1)

df = pd.merge(df, poly[['GID_1', 'NAME_1', 'GID_2', 'NAME_2']], on=['GID_2'], how='left')
df = df.rename({'GID_1': 'DISTRICT_ID',
                'NAME_1': 'DISTRICT_NAME',
                'GID_2': "MUNICIPALITY_ID",

                'NAME_2': "MUNICIPALITY_NAME"}, axis=1)
df.columns = df.columns.str.upper()
df = df.rename({"PM25_CONCENTRATION": "PM25",
                "PM10_CONCENTRATION": "PM10",
                "NO2_CONCENTRATION": "NO2"}, axis=1)

In [173]:
df

,MUNICIPALITY_ID,YEAR,HOSPITAL_ACCESIBILITY,SCHOOL_ACCESIBILITY,PM25,PM10,NO2,CO2E_20YR_EMISSIONS_QUANTITY,CH4_EMISSIONS_QUANTITY,TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY,AVG_D_MBPS_BROADBAND,KEY_STRUCTURES_WITHOUT_INTERNET,HEALTHCARE-FACILITIES-DIVERSITY,LUMINOSITY,AVG_D_MBPS_MOBILE,RAIL_LENGTH_FLOOD_RISK,ROAD_LENGTH_FLOOD_RISK,RAIL_LENGTH_HEATWAVE_RISK,ROAD_LENGTH_HEATWAVE_RISK,DISTRICT_ID,DISTRICT_NAME,MUNICIPALITY_NAME
0,SRB.1.1_1,2021,68.0,67.0,8.339596,11.149546,2.653598,1.014963e+06,267.411777,0.0,56.760213,11.111111,1.388204,8406.669981,32.129829,0.0,0.0000,90.178562,2091.458435,SRB.1_1,Borski,Bor
1,SRB.1.1_1,2022,68.0,67.0,8.545606,10.125109,2.486569,2.528427e+05,270.543667,0.0,52.114375,0.000000,1.388204,4614.979995,36.962571,0.0,0.0000,90.177980,2267.766246,SRB.1_1,Borski,Bor
2,SRB.1.1_1,2023,68.0,67.0,8.003265,9.656753,2.532644,5.614135e+05,270.248959,0.0,70.101850,0.000000,1.388204,6226.329994,43.568551,0.0,0.0000,90.177980,2507.180571,SRB.1_1,Borski,Bor
3,SRB.1.1_1,2024,68.0,67.0,8.199900,8.778352,0.186570,5.600742e+05,271.194468,0.0,99.311692,0.000000,1.388204,9653.610001,59.829482,0.0,0.0000,90.177980,2560.840628,SRB.1_1,Borski,Bor
4,SRB.1.2_1,2021,0.0,0.0,8.230006,10.621915,3.599601,7.234743e+05,121.539696,0.0,46.579333,0.000000,0.000000,3257.810001,26.880468,0.0,206235.5369,0.000000,1193.493364,SRB.1_1,Borski,Kladovo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,SRB.9.3_1,2024,0.0,9.0,6.867846,7.880904,-0.404043,-1.466550e+05,477.899229,0.0,0.000000,0.000000,0.000000,4765.720001,44.626340,0.0,0.0000,0.000000,3260.747844,SRB.9_1,Moravički,Ivanjica
640,SRB.9.4_1,2021,1.0,3.0,8.496553,12.604442,2.623166,3.203621e+05,107.097148,0.0,0.000000,0.000000,0.000000,4607.030001,37.369589,0.0,0.0000,14.831077,1400.087125,SRB.9_1,Moravički,Lučani
641,SRB.9.4_1,2022,1.0,3.0,8.302285,10.655915,2.707540,-2.776789e+04,108.632973,0.0,0.000000,0.000000,0.000000,4264.489998,32.758064,0.0,0.0000,14.831077,1619.994138,SRB.9_1,Moravički,Lučani
642,SRB.9.4_1,2023,1.0,3.0,7.385581,9.693388,2.556668,-2.607153e+05,108.824211,0.0,0.000000,0.000000,0.000000,1914.559998,41.890408,0.0,0.0000,14.800398,1866.055030,SRB.9_1,Moravički,Lučani


In [176]:
df_score = df[['DISTRICT_ID', 'DISTRICT_NAME', 'MUNICIPALITY_ID', 'MUNICIPALITY_NAME']]

high_pos_dir = ['HOSPITAL_ACCESIBILITY', 'SCHOOL_ACCESIBILITY', 'AVG_D_MBPS_BROADBAND', 'HEALTHCARE-FACILITIES-DIVERSITY', 'LUMINOSITY', 'AVG_D_MBPS_MOBILE']
high_neg_dir = ['CO2E_20YR_EMISSIONS_QUANTITY', 'CH4_EMISSIONS_QUANTITY', 'TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY', 'KEY_STRUCTURES_WITHOUT_INTERNET', 
                'RAIL_LENGTH_FLOOD_RISK', 'ROAD_LENGTH_FLOOD_RISK', 'RAIL_LENGTH_HEATWAVE_RISK','ROAD_LENGTH_HEATWAVE_RISK',
                'PM25', 'PM10', 'NO2']

national_avg = {}

for var in df.drop(['DISTRICT_ID', 'DISTRICT_NAME', 'MUNICIPALITY_ID', 'MUNICIPALITY_NAME', 'YEAR'], axis=1).columns:

    national_avg[var] = np.round(df[var].mean(), 4)

    if var in high_pos_dir:
        df_score[var+'_SCORE'] = np.round(df[var].rank(pct=True, ascending=True) * 100, 2)
    if var in high_neg_dir:
        df_score[var+'_SCORE'] = np.round(df[var].rank(pct=True, ascending=False) * 100, 2)




df_score['ENERGY_ACCESS_SCORE'] = df_score['LUMINOSITY_SCORE']
df_score['DIGITALIZATION_SCORE'] = np.round((df_score['AVG_D_MBPS_BROADBAND_SCORE'] + df_score['AVG_D_MBPS_MOBILE_SCORE'] + df_score['KEY_STRUCTURES_WITHOUT_INTERNET_SCORE']) / 3, 2)
df_score['SUSTAINABLE_TRANSPORT_SCORE'] = np.round((df_score['RAIL_LENGTH_FLOOD_RISK_SCORE'] + df_score['ROAD_LENGTH_FLOOD_RISK_SCORE'] + df_score['RAIL_LENGTH_HEATWAVE_RISK_SCORE'] + df_score['ROAD_LENGTH_HEATWAVE_RISK_SCORE']) / 4, 2)
df_score['EDUCATION_SCORE'] = df_score['SCHOOL_ACCESIBILITY_SCORE']
df_score['HEALTH_SCORE'] = np.round((df_score['HOSPITAL_ACCESIBILITY_SCORE'] + df_score['HEALTHCARE-FACILITIES-DIVERSITY_SCORE']) / 2, 2)
df_score['ENVIRONMENT_SCORE'] = np.round((df_score['CO2E_20YR_EMISSIONS_QUANTITY_SCORE'] + df_score['TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY_SCORE'] + df_score['PM25_SCORE'] + df_score['PM10_SCORE'] + df_score['NO2_SCORE']) / 5, 2)

df_score['PROSPERITY_SCORE'] = np.round((df_score['ENERGY_ACCESS_SCORE'] + df_score['DIGITALIZATION_SCORE'] + df_score['SUSTAINABLE_TRANSPORT_SCORE']) / 3, 2)
df_score['LIVABILITY_SCORE'] = np.round((df_score['EDUCATION_SCORE'] + df_score['HEALTH_SCORE']+ df_score['ENVIRONMENT_SCORE']) / 3, 2)

In [177]:
df_score

,DISTRICT_ID,DISTRICT_NAME,MUNICIPALITY_ID,MUNICIPALITY_NAME,HOSPITAL_ACCESIBILITY_SCORE,SCHOOL_ACCESIBILITY_SCORE,PM25_SCORE,PM10_SCORE,NO2_SCORE,CO2E_20YR_EMISSIONS_QUANTITY_SCORE,CH4_EMISSIONS_QUANTITY_SCORE,TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY_SCORE,AVG_D_MBPS_BROADBAND_SCORE,KEY_STRUCTURES_WITHOUT_INTERNET_SCORE,HEALTHCARE-FACILITIES-DIVERSITY_SCORE,LUMINOSITY_SCORE,AVG_D_MBPS_MOBILE_SCORE,RAIL_LENGTH_FLOOD_RISK_SCORE,ROAD_LENGTH_FLOOD_RISK_SCORE,RAIL_LENGTH_HEATWAVE_RISK_SCORE,ROAD_LENGTH_HEATWAVE_RISK_SCORE,ENERGY_ACCESS_SCORE,DIGITALIZATION_SCORE,SUSTAINABLE_TRANSPORT_SCORE,EDUCATION_SCORE,HEALTH_SCORE,ENVIRONMENT_SCORE,PROSPERITY_SCORE,LIVABILITY_SCORE
0,SRB.1_1,Borski,SRB.1.1_1,Bor,89.83,85.17,81.99,62.73,69.57,6.52,35.09,51.32,66.15,23.68,96.04,82.76,11.96,58.85,68.71,11.34,19.88,82.76,33.93,39.70,85.17,92.94,54.43,52.13,77.51
1,SRB.1_1,Borski,SRB.1.1_1,Bor,89.83,85.17,77.48,81.37,74.07,22.20,34.78,51.32,63.35,64.60,96.04,66.61,23.14,58.85,68.71,11.72,16.77,66.61,50.36,39.01,85.17,92.94,61.29,51.99,79.80
2,SRB.1_1,Borski,SRB.1.1_1,Bor,89.83,85.17,88.35,85.87,73.60,13.04,34.94,51.32,73.45,64.60,96.04,74.69,40.99,58.85,68.71,11.72,11.65,74.69,59.68,37.73,85.17,92.94,62.44,57.37,80.18
3,SRB.1_1,Borski,SRB.1.1_1,Bor,89.83,85.17,85.09,94.10,97.20,13.20,34.63,51.32,86.49,64.60,96.04,87.58,79.35,58.85,68.71,11.49,11.18,87.58,76.81,37.56,85.17,92.94,68.18,67.32,82.10
4,SRB.1_1,Borski,SRB.1.2_1,Kladovo,30.51,13.12,84.63,73.60,51.55,9.78,62.73,51.32,58.23,64.60,27.10,49.07,3.42,58.85,7.30,84.16,54.04,49.07,42.08,51.09,13.12,28.80,54.18,47.41,32.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,SRB.9_1,Moravički,SRB.9.3_1,Ivanjica,30.51,51.94,94.88,98.14,99.53,76.71,21.58,51.32,7.14,64.60,27.10,67.39,43.17,58.85,68.71,84.16,5.12,67.39,38.30,54.21,51.94,28.80,84.12,53.30,54.95
640,SRB.9_1,Moravički,SRB.9.4_1,Lučani,62.19,37.97,79.04,36.34,70.65,18.01,68.63,51.32,7.14,64.60,27.10,66.30,24.07,58.85,68.71,57.22,41.61,66.30,31.94,56.60,37.97,44.64,51.07,51.61,44.56
641,SRB.9_1,Moravički,SRB.9.4_1,Lučani,62.19,37.97,82.61,72.67,68.48,63.98,67.24,51.32,7.14,64.60,27.10,63.20,13.04,58.85,68.71,57.22,33.70,63.20,28.26,54.62,37.97,44.64,67.81,48.69,50.14
642,SRB.9_1,Moravički,SRB.9.4_1,Lučani,62.19,37.97,91.77,85.56,72.83,85.09,66.93,51.32,7.14,64.60,27.10,26.40,35.40,58.85,68.71,57.53,28.26,26.40,35.71,53.34,37.97,44.64,77.31,38.48,53.31


In [178]:
national_avg

{'HOSPITAL_ACCESIBILITY': np.float64(19.0559),
 'SCHOOL_ACCESIBILITY': np.float64(26.1491),
 'PM25': np.float64(9.6004),
 'PM10': np.float64(12.0836),
 'NO2': np.float64(3.9593),
 'CO2E_20YR_EMISSIONS_QUANTITY': np.float64(254965.7142),
 'CH4_EMISSIONS_QUANTITY': np.float64(1096.072),
 'TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY': np.float64(140821.972),
 'AVG_D_MBPS_BROADBAND': np.float64(50.1977),
 'KEY_STRUCTURES_WITHOUT_INTERNET': np.float64(12.8594),
 'HEALTHCARE-FACILITIES-DIVERSITY': np.float64(0.4135),
 'LUMINOSITY': np.float64(5022.1899),
 'AVG_D_MBPS_MOBILE': np.float64(48.6052),
 'RAIL_LENGTH_FLOOD_RISK': np.float64(1803.1453),
 'ROAD_LENGTH_FLOOD_RISK': np.float64(47884.6725),
 'RAIL_LENGTH_HEATWAVE_RISK': np.float64(40.4833),
 'ROAD_LENGTH_HEATWAVE_RISK': np.float64(1466.9218)}

In [179]:
df_score.to_csv(r"D:\Work\WB\LDT\countries\SRB\datasets/SRB_full_scores.csv", index=False)

In [2]:
df = pd.read_csv("D:\Work\WB\LDT\countries\SRB\datasets\RS_processed-data_full-indicator-list.csv".replace('\\', '/'))
air = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\datasets\SRB_air_pollution_2022.csv")

poly = gpd.read_file(r"D:\Work\WB\LDT\countries\SRB\shapefiles\gadm41_SRB_2.json")
df.columns

Index(['GID_2', 'NAME_2', 'year', 'hospital_accesibility',
       'school_accesibility', 'pm25_concentration', 'o3_concentration',
       'co2e_20yr_emissions_quantity', 'co2e_20yr_emissions_factor',
       'agriculture_emissions', 'forestry-and-land-use_emissions',
       'total_powerplant_coal_emissions_quantity',
       'assets_broadband_speed_target', 'avg_d_mbps_broadband',
       'key_structures_without_internet', 'healthcare-facilities-diversity',
       'luminosity', 'avg_d_mbps_mobile', 'railway_length_flood_risk',
       'road_length_flood_risk', 'population_sum', 'rail_length_flood_pcap',
       'road_length_flood_pcap', 'rail_length_heatwave_risk',
       'road_length_heatwave_risk', 'rail_length_pcap_heatwave',
       'road_length_pcap_heatwave'],
      dtype='object')

In [3]:
df = df.drop(['co2e_20yr_emissions_factor',
       'agriculture_emissions', 'forestry-and-land-use_emissions',
       'assets_broadband_speed_target',
       'rail_length_flood_pcap',
       'road_length_flood_pcap', 'population_sum',
       'rail_length_pcap_heatwave',
       'road_length_pcap_heatwave',
       'pm25_concentration', 'o3_concentration'], axis=1)

df = pd.merge(df, air, on=['GID_2', 'year'], how='left')
df = pd.merge(df, poly[['GID_1', 'NAME_1', 'GID_2', 'NAME_2']], on=['GID_2', 'NAME_2'], how='left')
df = df.rename({'GID_1': 'DISTRICT_ID',
                'NAME_1': 'DISTRICT_NAME',
                'GID_2': "MUNICIPALITY_ID",
                'NAME_2': "MUNICIPALITY_NAME"}, axis=1)
df.columns = df.columns.str.upper()

In [4]:
df.columns

Index(['MUNICIPALITY_ID', 'MUNICIPALITY_NAME', 'YEAR', 'HOSPITAL_ACCESIBILITY',
       'SCHOOL_ACCESIBILITY', 'CO2E_20YR_EMISSIONS_QUANTITY',
       'TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY', 'AVG_D_MBPS_BROADBAND',
       'KEY_STRUCTURES_WITHOUT_INTERNET', 'HEALTHCARE-FACILITIES-DIVERSITY',
       'LUMINOSITY', 'AVG_D_MBPS_MOBILE', 'RAILWAY_LENGTH_FLOOD_RISK',
       'ROAD_LENGTH_FLOOD_RISK', 'RAIL_LENGTH_HEATWAVE_RISK',
       'ROAD_LENGTH_HEATWAVE_RISK', 'PM25', 'PM10', 'NO2', 'DISTRICT_ID',
       'DISTRICT_NAME'],
      dtype='object')

In [5]:
df_score = df[['DISTRICT_ID', 'DISTRICT_NAME', 'MUNICIPALITY_ID', 'MUNICIPALITY_NAME']]

high_pos_dir = ['HOSPITAL_ACCESIBILITY', 'SCHOOL_ACCESIBILITY', 'AVG_D_MBPS_BROADBAND', 'HEALTHCARE-FACILITIES-DIVERSITY', 'LUMINOSITY', 'AVG_D_MBPS_MOBILE']
high_neg_dir = ['CO2E_20YR_EMISSIONS_QUANTITY', 'TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY', 'KEY_STRUCTURES_WITHOUT_INTERNET', 
                'RAILWAY_LENGTH_FLOOD_RISK', 'ROAD_LENGTH_FLOOD_RISK', 'RAIL_LENGTH_HEATWAVE_RISK','ROAD_LENGTH_HEATWAVE_RISK',
                'PM25', 'PM10', 'NO2']

national_avg = {}

for var in df.drop(['DISTRICT_ID', 'DISTRICT_NAME', 'MUNICIPALITY_ID', 'MUNICIPALITY_NAME', 'YEAR'], axis=1).columns:

    national_avg[var] = np.round(df[var].mean(), 4)

    if var in high_pos_dir:
        df_score[var+'_SCORE'] = np.round(df[var].rank(pct=True, ascending=True) * 100, 2)
    if var in high_neg_dir:
        df_score[var+'_SCORE'] = np.round(df[var].rank(pct=True, ascending=False) * 100, 2)




df_score['ENERGY_ACCESS_SCORE'] = df_score['LUMINOSITY_SCORE']
df_score['DIGITALIZATION_SCORE'] = np.round((df_score['AVG_D_MBPS_BROADBAND_SCORE'] + df_score['AVG_D_MBPS_MOBILE_SCORE'] + df_score['KEY_STRUCTURES_WITHOUT_INTERNET_SCORE']) / 3, 2)
df_score['SUSTAINABLE_TRANSPORT_SCORE'] = np.round((df_score['RAILWAY_LENGTH_FLOOD_RISK_SCORE'] + df_score['ROAD_LENGTH_FLOOD_RISK_SCORE'] + df_score['RAIL_LENGTH_HEATWAVE_RISK_SCORE'] + df_score['ROAD_LENGTH_HEATWAVE_RISK_SCORE']) / 4, 2)
df_score['EDUCATION_SCORE'] = df_score['SCHOOL_ACCESIBILITY_SCORE']
df_score['HEALTH_SCORE'] = np.round((df_score['HOSPITAL_ACCESIBILITY_SCORE'] + df_score['HEALTHCARE-FACILITIES-DIVERSITY_SCORE']) / 2, 2)
df_score['ENVIRONMENT_SCORE'] = np.round((df_score['CO2E_20YR_EMISSIONS_QUANTITY_SCORE'] + df_score['TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY_SCORE'] + df_score['PM25_SCORE'] + df_score['PM10_SCORE'] + df_score['NO2_SCORE']) / 5, 2)

df_score['PROSPERITY_SCORE'] = np.round((df_score['ENERGY_ACCESS_SCORE'] + df_score['DIGITALIZATION_SCORE'] + df_score['SUSTAINABLE_TRANSPORT_SCORE']) / 3, 2)
df_score['LIVABILITY_SCORE'] = np.round((df_score['EDUCATION_SCORE'] + df_score['HEALTH_SCORE']+ df_score['ENVIRONMENT_SCORE']) / 3, 2)

In [8]:
national_avg

{'HOSPITAL_ACCESIBILITY': np.float64(19.0559),
 'SCHOOL_ACCESIBILITY': np.float64(26.1491),
 'CO2E_20YR_EMISSIONS_QUANTITY': np.float64(73556.1049),
 'TOTAL_POWERPLANT_COAL_EMISSIONS_QUANTITY': np.float64(152540.3727),
 'AVG_D_MBPS_BROADBAND': np.float64(42.1856),
 'KEY_STRUCTURES_WITHOUT_INTERNET': np.float64(9.8683),
 'HEALTHCARE-FACILITIES-DIVERSITY': np.float64(0.4135),
 'LUMINOSITY': np.float64(49624.2108),
 'AVG_D_MBPS_MOBILE': np.float64(44.0529),
 'RAILWAY_LENGTH_FLOOD_RISK': np.float64(1887.1511),
 'ROAD_LENGTH_FLOOD_RISK': np.float64(54723.7725),
 'RAIL_LENGTH_HEATWAVE_RISK': np.float64(63.0746),
 'ROAD_LENGTH_HEATWAVE_RISK': np.float64(1746.8355),
 'PM25': np.float64(9.9984),
 'PM10': np.float64(12.4113),
 'NO2': np.float64(4.622)}

In [9]:
df.to_csv(r"D:\Work\WB\LDT\countries\SRB\datasets/SRB_full_absolute.csv", index=False)
df_score.to_csv(r"D:\Work\WB\LDT\countries\SRB\datasets/SRB_full_scores.csv", index=False)

In [1]:
1887.1511 / 1000

1.8871511

In [2]:
54723.7725 / 1000

54.7237725